In [1]:
# ── Cell 2: Imports ───────────────────────────
from ultralytics import YOLO
import torch
import gc
import os
import shutil
import yaml
from pathlib import Path

print('✅ Imports done')

✅ Imports done


In [2]:
# ── Cell 3: GPU Check ─────────────────────────
torch.cuda.empty_cache()
gc.collect()

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU — training will be slow on CPU')

CUDA available: True
GPU:  NVIDIA GeForce RTX 4060
VRAM: 8.3 GB


In [3]:
# ── Cell 4: Config ────────────────────────────
MODEL_SIZE   = 's'
EPOCHS       = 50
IMG_SIZE     = 640     # 8GB VRAM limit — don't increase
BATCH        = 8       # 8GB VRAM — don't increase
PROJECT      = './runs'
RUN_NAME     = 'yolov8s_local'
SAVE_DIR     = './saved_weights'

# ── Update this if your dataset folder name is different
DATASET_ROOT = './archive/vehicles'

print(f'Model:    YOLOv8{MODEL_SIZE}')
print(f'Epochs:   {EPOCHS}')
print(f'ImgSize:  {IMG_SIZE}')
print(f'Batch:    {BATCH}')
print(f'Dataset:  {DATASET_ROOT}')

Model:    YOLOv8s
Epochs:   50
ImgSize:  640
Batch:    8
Dataset:  ./archive/vehicles


In [4]:
# ── Cell 5: Create data.yaml ──────────────────
dataset_abs = str(Path(DATASET_ROOT).resolve())

yaml_data = {
    'path' : dataset_abs,
    'train': 'train/images',
    'val'  : 'val/images',
    'nc'   : 5,
    'names': ['bus', 'car', 'pickup', 'truck', 'van']
}

yaml_path = './data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

print(f'✅ data.yaml saved')
print(f'   Dataset path: {dataset_abs}')

# Verify dataset folders exist
for split in ['train/images', 'train/labels', 'val/images', 'val/labels']:
    full = Path(DATASET_ROOT) / split
    status = '✅' if full.exists() else '❌ NOT FOUND'
    print(f'   {status}  {split}')

✅ data.yaml saved
   Dataset path: /home/rashedul/project/YOLO_PROJECT/archive/vehicles
   ✅  train/images
   ✅  train/labels
   ✅  val/images
   ✅  val/labels


In [5]:
# ── Cell 6: Load Model ────────────────────────
model = YOLO(f'yolov8{MODEL_SIZE}.pt')
print(f'✅ Loaded YOLOv8{MODEL_SIZE}')
print(f'   Parameters: {sum(p.numel() for p in model.model.parameters()):,}')

✅ Loaded YOLOv8s
   Parameters: 11,166,560


In [6]:
# ── Cell 7: Train ─────────────────────────────
print(f'🏋️ Training: {EPOCHS} epochs | imgsz={IMG_SIZE} | batch={BATCH}\n')

results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=0,
    cache='disk',
    workers=4,
    patience=15,
    save=True,
    save_period=10,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    verbose=True,
    seed=42,
    deterministic=False,

    # Learning Rate
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,

    # Scheduler
    cos_lr=True,
    close_mosaic=10,

    # Loss
    box=9.0,
    cls=0.5,
    dfl=1.5,

    # Memory
    amp=True,
    fraction=1.0,
    rect=False,
    single_cls=False,
    profile=False,

    # Augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.6,
    shear=2.0,
    perspective=0.0001,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
)

print('\n✅ Training complete!')

🏋️ Training: 50 epochs | imgsz=640 | batch=8

New https://pypi.org/project/ultralytics/8.4.177 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.176 🚀 Python-3.14.8 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 7886MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=9.0, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=./data.yaml, degrees=5.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1,

KeyboardInterrupt: 

In [10]:
# ── Cell 9: Cleanup
del model
torch.cuda.empty_cache()
gc.collect()
print('✅ GPU memory cleared')

✅ GPU memory cleared


In [13]:
# ── Cell 10: Standard Validation ─────────────
from ultralytics import YOLO
import torch
from pathlib import Path

PROJECT  = './runs'
RUN_NAME = 'yolov8s_local'
IMG_SIZE = 640
yaml_path = './data.yaml'

# Auto-find best.pt — handles both possible paths
candidates = [
    f'{PROJECT}/{RUN_NAME}/weights/best.pt',
    f'{PROJECT}/detect/{RUN_NAME}/weights/best.pt',
]

best_src = None
for p in candidates:
    if Path(p).exists():
        best_src = p
        break

if best_src is None:
    # Search recursively as fallback
    matches = list(Path(PROJECT).rglob('best.pt'))
    if matches:
        best_src = str(matches[0])
        print(f'⚠️  Found best.pt at: {best_src}')
    else:
        raise FileNotFoundError(f'best.pt not found under {PROJECT}/')

print(f'📂 Loading: {best_src}')
best_model = YOLO(best_src)
print('✅ Loaded best model\n')

val_results = best_model.val(data=yaml_path, imgsz=IMG_SIZE, device=0)

print(f'\n📊 Standard Results:')
print(f'   mAP50:     {val_results.box.map50:.4f}')
print(f'   mAP50-95:  {val_results.box.map:.4f}')
print(f'   Precision: {val_results.box.mp:.4f}')
print(f'   Recall:    {val_results.box.mr:.4f}')

⚠️  Found best.pt at: runs/detect/runs/yolov8s_local/weights/best.pt
📂 Loading: runs/detect/runs/yolov8s_local/weights/best.pt
✅ Loaded best model

Ultralytics 8.4.176 🚀 Python-3.14.8 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 7886MiB)
Model summary (fused): 72 layers, 11,127,519 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.2±0.1 ms, read: 757.3±549.2 MB/s, size: 476.4 KB)
val: Scanning /home/rashedul/project/YOLO_PROJECT/archive/vehicles/val/labels.cache... 7431 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 7431/7431 3.5Git/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 37, len(boxes) = 16299. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 465/465 6.5it/s 1:12<0.2sss
            

In [15]:
# ── Cell 11: TTA Validation ─────────────────
# augment=True gives free ~1% mAP boost
tta_results = best_model.val(data=yaml_path, imgsz=IMG_SIZE, device=0, augment=True)

print(f'📊 TTA Results:')
print(f'   mAP50:     {tta_results.box.map50:.4f}')
print(f'   mAP50-95:  {tta_results.box.map:.4f}')
print(f'   Precision: {tta_results.box.mp:.4f}')
print(f'   Recall:    {tta_results.box.mr:.4f}')

baseline = 0.8968
gain = tta_results.box.map50 - baseline
print(f'\n🎯 Gain over Kaggle baseline (0.8968): {gain:+.4f}')

if tta_results.box.map50 >= 0.90:
    print('   ✅ TARGET HIT — 90%+ mAP50 achieved!')
elif tta_results.box.map50 >= 0.88:
    print('   🔶 Almost there! Run more epochs to push past 90%')
else:
    print('   ⚠️  Keep training — more epochs needed')

Ultralytics 8.4.176 🚀 Python-3.14.8 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 7886MiB)
Model summary (fused): 72 layers, 11,127,519 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 305.6±113.3 MB/s, size: 102.7 KB)
val: Scanning /home/rashedul/project/YOLO_PROJECT/archive/vehicles/val/labels.cache... 7431 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 7431/7431 3.1Git/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 37, len(boxes) = 16299. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 465/465 5.0it/s 1:320.2sss
                   all       7431      16299      0.858      0.835      0.901      0.701
                   bus       1967       3248      0.851      0.807      

In [ ]:
# ── Cell 12: YOLOv10s Train ───────────────────
from ultralytics import YOLO
import torch, gc, os, shutil, yaml
from pathlib import Path

RUN_NAME_V10 = 'yolov10s_local'
MODEL_SIZE   = 's'
EPOCHS       = 50
IMG_SIZE     = 640
BATCH        = 8
PROJECT      = './runs'
SAVE_DIR     = './saved_weights'
yaml_path    = './data.yaml'

torch.cuda.empty_cache()
gc.collect()
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB\n')

model_v10 = YOLO(f'yolov10{MODEL_SIZE}.pt')
print(f'✅ Loaded YOLOv10{MODEL_SIZE}')
print(f'   Parameters: {sum(p.numel() for p in model_v10.model.parameters()):,}\n')

print(f'🏋️ Training: {EPOCHS} epochs | imgsz={IMG_SIZE} | batch={BATCH}\n')

results_v10 = model_v10.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=0,
    cache='disk',
    workers=4,
    patience=15,
    save=True,
    save_period=10,
    project=PROJECT,
    name=RUN_NAME_V10,
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    verbose=True,
    seed=42,
    deterministic=False,

    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    cos_lr=True,
    close_mosaic=10,

    box=7.5,
    cls=0.5,
    dfl=1.5,

    amp=True,
    fraction=1.0,
    rect=False,
    single_cls=False,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.6,
    shear=2.0,
    perspective=0.0001,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
)

print('\n✅ YOLOv10s training complete!')

os.makedirs(SAVE_DIR, exist_ok=True)

# Auto-find best.pt
matches = list(Path(PROJECT).rglob(f'*{RUN_NAME_V10}*/weights/best.pt'))
best_v10 = str(matches[0]) if matches else f'{PROJECT}/{RUN_NAME_V10}/weights/best.pt'
last_matches = list(Path(PROJECT).rglob(f'*{RUN_NAME_V10}*/weights/last.pt'))
last_v10 = str(last_matches[0]) if last_matches else f'{PROJECT}/{RUN_NAME_V10}/weights/last.pt'

shutil.copy(best_v10, f'{SAVE_DIR}/yolov10s_best.pt')
shutil.copy(last_v10, f'{SAVE_DIR}/yolov10s_last.pt')
print(f'💾 Saved → {SAVE_DIR}/yolov10s_best.pt')
print(f'   Found at: {best_v10}')

GPU: NVIDIA GeForce RTX 4060
VRAM: 8.3 GB

✅ Loaded YOLOv10s
   Parameters: 8,128,272

🏋️ Training: 50 epochs | imgsz=640 | batch=8

New https://pypi.org/project/ultralytics/8.4.177 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.176 🚀 Python-3.14.8 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 7886MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=./data.yaml, degrees=5.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.

In [ ]:
# ── Cell 13: YOLOv10s Validate ────────────────
from ultralytics import YOLO
from pathlib import Path

PROJECT      = './runs'
RUN_NAME_V10 = 'yolov10s_local'
IMG_SIZE     = 640
yaml_path    = './data.yaml'

import torch, gc
del model_v10
torch.cuda.empty_cache()
gc.collect()

# Auto-find best.pt
matches = list(Path(PROJECT).rglob(f'*{RUN_NAME_V10}*/weights/best.pt'))
best_v10 = str(matches[0]) if matches else f'{PROJECT}/{RUN_NAME_V10}/weights/best.pt'
print(f'📂 Loading: {best_v10}')

m_v10 = YOLO(best_v10)

val_v10 = m_v10.val(data=yaml_path, imgsz=IMG_SIZE, device=0)
print(f'\n📊 YOLOv10s Standard:')
print(f'   mAP50:     {val_v10.box.map50:.4f}')
print(f'   mAP50-95:  {val_v10.box.map:.4f}')
print(f'   Precision: {val_v10.box.mp:.4f}')
print(f'   Recall:    {val_v10.box.mr:.4f}')

tta_v10 = m_v10.val(data=yaml_path, imgsz=IMG_SIZE, device=0, augment=True)
print(f'\n📊 YOLOv10s TTA:')
print(f'   mAP50:     {tta_v10.box.map50:.4f}')
print(f'   mAP50-95:  {tta_v10.box.map:.4f}')
print(f'   Precision: {tta_v10.box.mp:.4f}')
print(f'   Recall:    {tta_v10.box.mr:.4f}')

baseline = 0.8968
gain = tta_v10.box.map50 - baseline
print(f'\n🎯 Gain over YOLOv8s baseline (0.8968): {gain:+.4f}')
print('✅ TARGET HIT!' if tta_v10.box.map50 >= 0.90 else '🔶 Try YOLO11s next')

In [ ]:
# ── Cell 14: YOLO11s Train ────────────────────
del m_v10
torch.cuda.empty_cache()
gc.collect()

RUN_NAME_V11 = 'yolo11s_local'

# Note: no 'v' in the name — it's yolo11s.pt not yolov11s.pt
model_v11 = YOLO('yolo11s.pt')
print(f'✅ Loaded YOLO11s')
print(f'   Parameters: {sum(p.numel() for p in model_v11.model.parameters()):,}\n')

results_v11 = model_v11.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=0,
    cache='disk',
    workers=4,
    patience=15,
    save=True,
    save_period=10,
    project=PROJECT,
    name=RUN_NAME_V11,
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    verbose=True,
    seed=42,
    deterministic=False,

    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    cos_lr=True,
    close_mosaic=10,

    box=7.5,
    cls=0.5,
    dfl=1.5,

    amp=True,
    fraction=1.0,
    rect=False,
    single_cls=False,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.6,
    shear=2.0,
    perspective=0.0001,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
)

print('\n✅ YOLO11s training complete!')

os.makedirs(SAVE_DIR, exist_ok=True)
best_v11 = f'{PROJECT}/{RUN_NAME_V11}/weights/best.pt'
last_v11 = f'{PROJECT}/{RUN_NAME_V11}/weights/last.pt'
shutil.copy(best_v11, f'{SAVE_DIR}/yolo11s_best.pt')
shutil.copy(last_v11, f'{SAVE_DIR}/yolo11s_last.pt')
print(f'💾 Saved to {SAVE_DIR}/yolo11s_best.pt')

In [ ]:
# ── Cell 15: YOLO11s Validate + Comparison ────
del model_v11
torch.cuda.empty_cache()
gc.collect()

best_v11 = f'{PROJECT}/{RUN_NAME_V11}/weights/best.pt'
m_v11 = YOLO(best_v11)

val_v11 = m_v11.val(data=yaml_path, imgsz=IMG_SIZE, device=0)
print(f'📊 YOLO11s Standard:  mAP50={val_v11.box.map50:.4f}  mAP50-95={val_v11.box.map:.4f}')

tta_v11 = m_v11.val(data=yaml_path, imgsz=IMG_SIZE, device=0, augment=True)
print(f'📊 YOLO11s TTA:       mAP50={tta_v11.box.map50:.4f}  mAP50-95={tta_v11.box.map:.4f}')

# ── Final comparison ──────────────────────────
print('\n' + '='*50)
print('📊 FINAL COMPARISON (TTA mAP50)')
print('='*50)

# load v8 result from Cell 11 (re-run if needed)
best_v8 = f'{PROJECT}/{RUN_NAME}/weights/best.pt'
m_v8 = YOLO(best_v8)
tta_v8 = m_v8.val(data=yaml_path, imgsz=IMG_SIZE, device=0, augment=True)

scores = {
    'YOLOv8s  (baseline)': 0.8968,
    'YOLOv8s  (this run)': tta_v8.box.map50,
    'YOLOv10s (this run)': tta_v10.box.map50,
    'YOLO11s  (this run)': tta_v11.box.map50,
}

for name, score in scores.items():
    bar = '✅' if score >= 0.90 else '🔶' if score >= 0.88 else '  '
    print(f'  {bar}  {name}: {score:.4f}')

best_score = max(tta_v8.box.map50, tta_v10.box.map50, tta_v11.box.map50)
print(f'\n🏆 Best model: {best_score:.4f}')
if best_score >= 0.90:
    print('   ✅ 90%+ TARGET ACHIEVED!')
else:
    print(f'   Gap to 90%: {0.90 - best_score:.4f} — try model size "m"')